<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Tomás Vallejo
- Nombre de alumno 2:


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [1]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 126 packages in 516ms                                       
⠙ Preparing packages... (0/5)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/5)-------------------     0 B/76.54 KiB           
⠙ Preparing packages... (0/5)------------------- 16.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/5)m------------------ 32.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/5)30m------------ 48.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/5)30m------------ 48.00 KiB/76.54 KiB         
pynndescent          ------------------------------     0 B/71.79 KiB
⠙ Preparing packages... (0/5)30m------------ 48.00 KiB/76.54 KiB         
pynndescent          ------------------------------     0 B/71.79 KiB
⠙ Preparing packages... (0/5)---------- 64.00 KiB/76.54 KiB         
pynndescent          ------------------------------ 16.00 KiB/71.79 KiB
⠙ Preparing packages... (0/5)---

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [ ]:
RANDOM_STATE = 20968061  # Semilla para reproducibilidad: mi rut sin dígito verificador


def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    # Revenue (ganancias por línea de transacción)
    df["Revenue"] = df["Quantity"] * df["Price"]

    # Fecha de referencia global para Recency
    max_date = df["InvoiceDate"].max()

    # Consolidar cada visita (Invoice): fecha mínima y revenue total, agrupamos por cliente e invoice
    invoice_df = (
        df.groupby(["Customer ID", "Invoice"])
        .agg(
            InvoiceDate=(
                "InvoiceDate",
                "min",
            ),  # Fecha de la visita: la fecha más temprana de las líneas que componen la visita (deberían ser todas iguales, pero por si acaso)
            InvoiceRevenue=("Revenue", "sum"),  # Sumamos el revenue de cada línea para obtener el total por visita
        )
        .reset_index()
        .sort_values(["Customer ID", "InvoiceDate"])  # Ordenamos por cliente y fecha para facilitar el cálculo de IVT
    )

    # IVT: días entre visitas consecutivas del mismo cliente
    invoice_df["IVT"] = invoice_df.groupby("Customer ID")["InvoiceDate"].diff().dt.days

    # Nuevo dataframe agregado por cliente
    agg = invoice_df.groupby("Customer ID").agg(
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max"),
        Frequency=("Invoice", "count"),
        TotalRevenue=("InvoiceRevenue", "sum"),
        Periodicity=("IVT", "std"),
    )

    # L, R, M desde el agregado
    agg["Length"] = (agg["last_date"] - agg["first_date"]).dt.days
    agg["Recency"] = (max_date - agg["last_date"]).dt.days
    agg["Monetary"] = (agg["TotalRevenue"] / agg["Frequency"]).round(2)

    # Un solo IVT retorna NaN (Al calcular con std()) se reemplaza por 0
    agg["Periodicity"] = agg["Periodicity"].fillna(0).round(0)

    return agg[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].reset_index()

In [10]:
features_df = custom_features(df_retail)
features_df.head()

,Customer ID,Length,Recency,Frequency,Monetary,Periodicity
0,12346.0,196,164,11,33.90,37.0
1,12347.0,37,2,2,661.66,0.0
2,12348.0,0,73,1,222.16,0.0
3,12349.0,181,42,3,890.38,102.0
4,12351.0,0,10,1,300.93,0.0


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [11]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Guarda el min y max de cada columna para usarlos en transform
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X):
        # Aplica la fórmula MinMax columna por columna
        return (X - self.min_) / (self.max_ - self.min_)

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [ ]:
# Importamos las clases necesarias
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from umap import UMAP

LRMFP_COLS = [
    "Length",
    "Recency",
    "Frequency",
    "Monetary",
    "Periodicity",
]  # 5 columnas de características, sin incluir "Customer ID"

# ColumnTransformer aplica MinMax sobre las 5 columnas LRMFP, descarta Customer ID
ct = ColumnTransformer(
    transformers=[("minmax", MinMax(), LRMFP_COLS)],
)

# Pipeline para cada técnica de reducción de dimensionalidad: primero extrae las características, luego las escala con MinMax, y finalmente aplica PCA, t-SNE o UMAP
pipeline_pca = Pipeline(
    steps=[
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ct),
        ("pca", PCA(n_components=2, random_state=RANDOM_STATE)),
    ]
)

pipeline_tsne = Pipeline(
    steps=[
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ct),
        ("tsne", TSNE(n_components=2, random_state=RANDOM_STATE)),
    ]
)

pipeline_umap = Pipeline(
    steps=[
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ct),
        (
            "umap",
            UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=20, min_dist=0.15),
        ),  # Utilizamos según la clase 11 esos hiperparámetros para UMAP
    ]
)

In [56]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

/Users/tomasvallejo/Documents/Universidad/Otoño 2026/MDS7202/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [57]:
# Extraer el PCA ya entrenado desde el pipeline
pca_step = pipeline_pca["pca"]

# Loadings: shape (n_features, n_components) → cada fila es una feature, cada columna una PC
loadings = pca_step.components_.T
loadings_df = pd.DataFrame(loadings, index=LRMFP_COLS, columns=["PC1", "PC2"]).round(3)

print(f"PC1: {pca_step.explained_variance_ratio_[0] * 100:.1f}%")
print(f"PC2: {pca_step.explained_variance_ratio_[1] * 100:.1f}%")
print(f"Total: {pca_step.explained_variance_ratio_.sum() * 100:.1f}%")

print("\nLoadings:")
print(loadings_df)

PC1: 73.6%
PC2: 19.8%
Total: 93.4%

Loadings:
               PC1    PC2
Length       0.863  0.446
Recency     -0.464  0.885
Frequency    0.045  0.017
Monetary     0.005 -0.002
Periodicity  0.196  0.129


In [17]:
# Gráfico de barras de loadings por componente
fig_loadings = make_subplots(rows=1, cols=2, subplot_titles=("PC1", "PC2"))

for i, pc in enumerate(["PC1", "PC2"]):
    fig_loadings.add_trace(
        go.Bar(
            x=LRMFP_COLS,
            y=loadings_df[pc],
            name=pc,
            marker_color=["crimson" if v < 0 else "steelblue" for v in loadings_df[pc]],
            showlegend=False,
        ),
        row=1,
        col=i + 1,
    )

fig_loadings.update_layout(
    title="Loadings de las dos primeras componentes principales (PCA)",
    height=400,
)
fig_loadings.update_yaxes(range=[-1, 1])
fig_loadings.show()

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Los loadings son los coeficientes que indican cuánto contribuye cada feature original a cada componente principal. Un valor alto en valor absoluto significa que esa feature tiene mucha influencia en esa componente, y el signo indica la dirección de esa contribución.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: PC1 explica el 73.6% de la varianza y está dominada por Length (0.863) y Recency (-0.464), lo que indica que esta componente principalmente separa clientes con larga trayectoria de compra de aquellos que llevan mucho tiempo sin comprar. PC2 explica el 19.8% adicional y está dominada casi exclusivamente por Recency (0.885), capturando una dimensión independiente de qué tan reciente fue la última compra. Frequency y Monetary tienen loadings muy cercanos a cero en ambas componentes, lo que sugiere que estas features aportan poca varianza al dataset y no son las que más diferencian a los clientes.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: Length y Recency apuntan en direcciones opuestas en PC1 (0.863 vs -0.464), lo cual tiene sentido de negocio: un cliente con larga trayectoria tiende a haber comprado recientemente, mientras que uno con alta recency lleva mucho tiempo sin volver. Frequency y Monetary apuntan en direcciones similares y ambas cercanas a cero, sugiriendo que están correlacionadas entre sí pero como se dijo antes, no son las fuentes principales de variación en este dataset.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [22]:
from sklearn.cluster import KMeans

# Pipeline igual al de t-SNE pero reemplazando la reducción por KMeans
inercias = []
for k in range(1, 21):
    pipeline_kmeans = Pipeline(
        steps=[
            ("features", FunctionTransformer(custom_features)),
            ("scaler", ct),
            ("kmeans", KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")),
        ]
    )
    pipeline_kmeans.fit(df_retail)
    inercias.append({"n_clusters": k, "inertia": pipeline_kmeans["kmeans"].inertia_})

inercias_df = pd.DataFrame(inercias)

# Gráfico del codo
px.line(
    inercias_df,
    x="n_clusters",
    y="inertia",
    title="Método del Codo - KMeans",
    labels={"n_clusters": "Número de Clusters (k)", "inertia": "Inercia"},
    markers=True,
).show()

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: Escogería k=6, ya que es el punto donde la reducción de inercia comienza a disminuir notoriamente. Entre k=5 y k=6 la inercia baja 24.7 puntos, mientras que de k=6 a k=7 la reducción cae a solo 14.9, indicando que agregar más clusters genera retornos decrecientes. 

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: Parcialmente. Si bien al analizar las reducciones concretas de inercia se puede identificar k=6 como punto de quiebre (reducción de 24.7 vs 14.9 del siguiente), visualmente la curva no presenta un codo brusco y evidente sino que decrece de forma gradual, lo que hace que la elección no sea inmediata solo mirando el gráfico.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: Como se vio en clases, el coeficiente de Silhouette es una buena alternativa ya que mide qué tan bien separados están los clusters comparando la cohesión interna con la separación entre ellos, entregando un valor entre -1 y 1 donde más cercano a 1 es mejor. También se podría usar BIC/AIC si se optara por un modelo probabilístico como GMM, el cual penaliza la complejidad del modelo evitando elegir un k innecesariamente alto.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [34]:
# Pipeline KMeans con k=6
pipeline_kmeans_final = Pipeline(
    steps=[
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ct),
        ("kmeans", KMeans(n_clusters=6, random_state=RANDOM_STATE, n_init="auto")),
    ]
)

pipeline_kmeans_final.fit(df_retail)
kmeans_labels = pipeline_kmeans_final["kmeans"].labels_

In [35]:
# Tabla de promedios por cluster
lrmfp_df = custom_features(df_retail).set_index("Customer ID")
lrmfp_df["Cluster"] = kmeans_labels

cluster_profile = (
    lrmfp_df.groupby("Cluster")[["Length", "Recency", "Frequency", "Monetary", "Periodicity"]].mean().round(1)
)

# Agregar conteo de clientes por cluster
cluster_profile["Count"] = lrmfp_df.groupby("Cluster").size()
print(cluster_profile)

         Length  Recency  Frequency  Monetary  Periodicity  Count
Cluster                                                          
0         322.5     21.4       11.5     425.5         29.8    915
1          17.1     42.7        1.7     377.9          2.2   1157
2           5.8    305.5        1.3     354.1          1.0    457
3         179.8     64.2        4.4     367.9         27.1    884
4          31.6    187.3        1.8     340.7          3.8    602
5         275.3     37.6        4.0     348.7        112.8    299


In [36]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=6", colors=kmeans_labels)

In [38]:
# Heatmap de perfiles por cluster
px.imshow(
    cluster_profile.drop(columns="Count"),
    color_continuous_scale="RdBu_r",
    text_auto=True,
    aspect="auto",
    title="Perfil medio por Cluster (KMeans K=6)",
    labels={"x": "Feature", "y": "Cluster", "color": "Media"},
    height=400,
).show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: En t-SNE y UMAP la separación es bastante clara, cada cluster ocupa una región diferenciada del espacio. En PCA la separación es menos limpia ya que algunos clusters se solapan visualmente, pero lo cual es esperable dado que PCA es lineal y no captura toda la estructura de los datos.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: Sí, especialmente en t-SNE y UMAP donde los grupos son visualmente compactos y bien separados. Los clusters reflejan perfiles de cliente con sentido de negocio real, como se puede ver en el heatmap.

- ¿Quedarían mejor más o menos clusters?

> Respuesta: k=6 parece adecuado. Agregar más clusters dividiría grupos que ya tienen perfiles similares sin aportar información nueva interpretable. Con menos clusters como k=5 (se realizó la prueba) se pierden distinciones relevantes como la separación entre clientes frecuentes y clientes irregulares de larga data.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: No necesariamente. Como se ve en las proyecciones, algunas agrupaciones tienen formas curvas e irregulares que KMeans no captura bien al asumir clusters esféricos. DBSCAN podría ser más adecuado ya que trabaja por densidad y no requiere definir k a priori, identificando además outliers naturalmente.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: 

- **C0 — Clientes VIP activos:** Length alto, Recency baja, Frequency alta y Monetary alto. Llevan mucho tiempo comprando, vuelven seguido y gastan bien.

- **C1 — Clientes nuevos esporádicos:** Length bajo, Recency media y Frequency muy baja. Llegaron hace poco y han comprado muy pocas veces.

- **C2 — Clientes perdidos:** Length muy bajo y Recency muy alta. Casi no tienen historia y llevan casi un año sin comprar.

- **C3 — Clientes ocasionales estables:** Length medio, Recency media y Periodicity media. Llevan un tiempo razonable pero compran poco y con cierta regularidad.

- **C4 — Clientes en riesgo:** Length bajo y Recency alta. Poca trayectoria y llevan varios meses sin volver.

- **C5 — Clientes irregulares veteranos:** Length alto y Periodicity muy alta. Llevan mucho tiempo como clientes pero con comportamiento de compra muy errático.


## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# Obtener datos escalados para aplicar DBSCAN
# Reutilizamos el pipeline hasta el scaler (sin reducción de dimensionalidad)
pipeline_scaler = Pipeline(
    steps=[
        ("features", FunctionTransformer(custom_features)),
        ("scaler", ct),
    ]
)
lrmfp_scaled = pipeline_scaler.fit_transform(df_retail)

# Gráfico de k-distancias para elegir eps
neighbors = NearestNeighbors(n_neighbors=2).fit(lrmfp_scaled)
distances, _ = neighbors.kneighbors(lrmfp_scaled)
distances = np.sort(distances[:, 1])

px.line(
    x=range(len(distances)),
    y=distances,
    labels={"x": "Puntos ordenados", "y": "Distancia al 2° vecino"},
    title="Gráfico de k-distancias (para elegir eps)",
).show()

In [52]:
dbscan = DBSCAN(eps=0.1, min_samples=5)
dbscan.fit(lrmfp_scaled)
dbscan_labels = dbscan.labels_

# Resumen de resultados
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_outliers = (dbscan_labels == -1).sum()
print(f"Clusters encontrados: {n_clusters} | Outliers (-1): {n_outliers}")
print(pd.Series(dbscan_labels).value_counts().sort_index())

Clusters encontrados: 1 | Outliers (-1): 49
-1      49
 0    4265
Name: count, dtype: int64


In [53]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

In [54]:
# Analizar quiénes son los outliers
lrmfp_df["DBSCAN"] = dbscan_labels
outliers_df = lrmfp_df[lrmfp_df["DBSCAN"] == -1][["Length", "Recency", "Frequency", "Monetary", "Periodicity"]]
print(f"\nPerfil promedio de outliers (n={n_outliers}):")
print(outliers_df.mean().round(1))


Perfil promedio de outliers (n=49):
Length          198.4
Recency          91.4
Frequency        24.4
Monetary       2916.3
Periodicity      56.6
dtype: float64


### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: Se aplicó DBSCAN sobre las características originales escaladas (LRMFP) en lugar de las proyecciones 2D, ya que PCA, t-SNE y UMAP sabemos que comprimen y distorsionan las distancias reales entre clientes al reducir dimensiones. Un cliente anómalo en el espacio de 5 features puede quedar visualmente cercano a clientes normales en la proyección 2D, lo que haría que DBSCAN no lo detecte correctamente. Por más que las proyecciones 2D intenten preservar la mayor cantidad de información, trabajar con los datos originales preserva TODA la información para una detección más confiable.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: Para elegir eps se usó el gráfico de k-distancias visto en clases, que ordena las distancias al segundo vecino más cercano de cada punto. El codo de la curva aparece alrededor de 0.05-0.15, por lo que se probaron valores en ese rango. Con eps=0.05 se obtuvieron 12 micro-clusters poco interpretables y 493 outliers. Con eps=0.15 solo se detectaron 15 outliers. El valor eps=0.1 resultó el más equilibrado con 1 cluster principal limpio y 49 outliers con un perfil claramente diferenciado. Para min_samples=5 se mantuvo el valor por defecto de sklearn, suficiente para filtrar grupos falsos.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: Sí, los outliers tienen una interpretación clara. Su Monetary promedio es de 2916.3, casi 7 veces mayor al del cliente típico, lo que indica que son clientes de alto valor económico. Sin embargo tienen una Recency de 91.4 días y una Periodicity de 56.6, lo que sugiere que compran de forma esporádica e irregular. Son clientes que cuando compran gastan mucho, pero no lo hacen frecuentemente ni con regularidad, lo que los aleja del comportamiento de la mayoría y los convierte en anomalías estadísticas con alto valor estratégico para el negocio.



# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>